# Módulo 07 · Redes y grafos — Soluciones de los ejercicios

Este notebook es **autocontenido**: reproduce la preparación mínima del notebook del módulo (club de kárate de Zachary, centralidades) y resuelve los cuatro ejercicios del final.

| Ejercicio | Pregunta | Sección |
|---|---|---|
| 1 | ¿A qué 3 personas contactar para difundir un mensaje? | 1 |
| 2 | ¿Qué pasa si eliminamos el nodo con mayor intermediación? | 2 |
| 3 | Adamic-Adar vs. vecinos comunes vs. Jaccard con 20 semillas | 3 |
| 4 | Centralidades en *Los miserables* | 4 |

Nota: en networkx 3.x el grafo del kárate trae un atributo `weight` (intensidad de la relación). Igual que en el notebook, las centralidades de grado, intermediación, cercanía y autovector se calculan **sin pesos** (comportamiento por defecto) y PageRank **con pesos** (también su comportamiento por defecto).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

G = nx.karate_club_graph()
pos = nx.spring_layout(G, seed=3)
faccion_real = np.array([0 if G.nodes[n]["club"] == "Mr. Hi" else 1 for n in G])

def dibujar(G, colores=None, tamanos=300, titulo="", ax=None, cmap="coolwarm", pos=pos):
    ax = ax or plt.gca()
    nx.draw_networkx(G, pos, node_color=colores if colores is not None else "lightblue", node_size=tamanos,
                     cmap=cmap, font_size=8, edge_color="lightgray", ax=ax)
    ax.set_title(titulo); ax.axis("off")

centralidades = pd.DataFrame({
    "grado": nx.degree_centrality(G),
    "intermediación": nx.betweenness_centrality(G),
    "cercanía": nx.closeness_centrality(G),
    "autovector": nx.eigenvector_centrality(G),
    "pagerank": nx.pagerank(G),
})
centralidades.sort_values("intermediación", ascending=False).head(6).round(3)

## Ejercicio 1 · Difundir un mensaje contactando a 3 personas

**Modelo de difusión (tal como lo pide el enunciado):** empezamos con 3 personas informadas («semillas»). En cada una de las 5 rondas, **cada persona informada** intenta contagiar a **cada vecino** no informado con probabilidad 0,3. Repetimos la simulación muchas veces (Monte Carlo) porque el resultado es aleatorio.

Comparamos varias estrategias:
- top-3 de cada centralidad (grado, intermediación, cercanía, autovector, PageRank);
- una estrategia **diversificada**: el mejor nodo de cada facción más el mejor «puente» que no sea redundante;
- 3 personas **al azar** (la referencia).

In [ ]:
def simular_difusion(G, semillas, p=0.3, rondas=5, rng=None):
    """Modelo SI en tiempo discreto: en cada ronda TODOS los informados intentan contagiar a sus vecinos."""
    informados = set(semillas)
    for _ in range(rondas):
        nuevos = set()
        for u in informados:
            for v in G.neighbors(u):
                if v not in informados and rng.random() < p:
                    nuevos.add(v)
        informados |= nuevos
    return len(informados)

def top3(medida):
    return list(centralidades[medida].sort_values(ascending=False).index[:3])

estrategias = {m: top3(m) for m in centralidades.columns}
# Diversificada: el líder de cada facción (0 y 33) + el nodo con más vecinos que NO son vecinos de 0 ni de 33
cubiertos = set(G[0]) | set(G[33]) | {0, 33}
extra = max((n for n in G if n not in (0, 33)), key=lambda n: len(set(G[n]) - cubiertos))
estrategias["diversificada"] = [0, 33, extra]
estrategias

In [ ]:
N_SIM = 2000
def evaluar(estrategias, rondas=5, p=0.3, n_sim=N_SIM, seed=42):
    rng = np.random.default_rng(seed)
    filas = []
    for nombre, semillas in estrategias.items():
        alcance = [simular_difusion(G, semillas, p, rondas, rng) for _ in range(n_sim)]
        filas.append({"estrategia": nombre, "semillas": semillas, "alcance medio": np.mean(alcance),
                      "% del club": np.mean(alcance) / len(G) * 100})
    alcance_azar = [simular_difusion(G, list(rng.choice(len(G), 3, replace=False)), p, rondas, rng) for _ in range(n_sim)]
    filas.append({"estrategia": "azar", "semillas": "3 aleatorias", "alcance medio": np.mean(alcance_azar),
                  "% del club": np.mean(alcance_azar) / len(G) * 100})
    return pd.DataFrame(filas).set_index("estrategia").round(2)

res_5 = evaluar(estrategias, rondas=5)
res_5

Con 5 rondas y $p = 0{,}3$ el club es tan pequeño y tan compacto (distancia media ≈ 2,4) que el mensaje llega a casi todo el mundo **sea cual sea** la estrategia: la diferencia entre elegir bien y elegir al azar es de unas 2–3 personas. Para ver mejor qué estrategia es más rápida, miramos el alcance tras **solo 2 rondas** (un mensaje con poco tiempo de vida, como una oferta flash):

In [ ]:
res_2 = evaluar(estrategias, rondas=2)
res_2

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
tabla_graf = pd.DataFrame({"2 rondas": res_2["% del club"], "5 rondas": res_5["% del club"]})
tabla_graf.plot.barh(ax=ax)
ax.set_xlabel("% medio del club informado"); ax.set_title("Alcance según la estrategia de semillas (2.000 simulaciones)")
plt.tight_layout(); plt.show()

Para cuantificar la incertidumbre de la comparación «mejor estrategia vs. azar», calculamos un intervalo bootstrap de la diferencia de medias con 2 rondas:

In [ ]:
rng = np.random.default_rng(7)
mejor = res_2.drop("azar")["alcance medio"].idxmax()
a = np.array([simular_difusion(G, estrategias[mejor], 0.3, 2, rng) for _ in range(N_SIM)])
b = np.array([simular_difusion(G, list(rng.choice(len(G), 3, replace=False)), 0.3, 2, rng) for _ in range(N_SIM)])
difs = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(1000)]
print(f"Mejor estrategia con 2 rondas: {mejor} {estrategias[mejor]}")
print(f"Diferencia media vs. azar: {a.mean() - b.mean():.2f} personas "
      f"(IC 95 % bootstrap: {np.percentile(difs, 2.5):.2f} a {np.percentile(difs, 97.5):.2f})")

**Interpretación y recomendación.**

- Con 5 rondas, todas las estrategias basadas en centralidad informan de media a **≈ 96–97 %** del club frente a **≈ 89 %** con 3 personas al azar: la red se satura y la ventaja es pequeña (unas 2–3 personas).
- Con 2 rondas la ventaja es enorme: **≈ 68–70 %** del club con semillas centrales frente a **≈ 40 %** al azar, es decir, unas **10 personas más** (IC 95 % bootstrap ≈ 10,1–10,5). Cuanto menos tiempo de vida tiene el mensaje, más importa elegir bien.
- Grado, intermediación y PageRank eligen las mismas semillas: **0 (instructor), 33 (presidente) y 32**; cercanía y autovector cambian 32 por 2. Las diferencias entre estas estrategias son de décimas de persona: en esta red, *casi cualquier* criterio de centralidad elige a los líderes de las dos facciones, que es lo importante.
- La estrategia «diversificada» (0, 33 y el nodo 31, que alcanza a más personas no cubiertas por los líderes) queda algo por debajo con 2 rondas (≈ 67 %): el nodo 31 tiene 6 contactos y el 32 tiene 12. La diversidad ya la garantizan 0 y 33 (uno en cada facción); para el tercer puesto pesa más el alcance propio.
- **Recomendación:** contactar a **0, 33 y 32** (top-3 por intermediación o por grado), con un «altavoz» en cada facción para no depender de los pocos puentes entre grupos. En una red grande (millones de clientes) conviene combinar centralidad con diversidad explícitamente (semillas en comunidades distintas), porque los *hubs* de una misma comunidad solapan su alcance.

*Variante opcional:* en el **modelo de cascada independiente** cada persona solo intenta contagiar una vez (en la ronda siguiente a enterarse). Es más realista para mensajes que se reenvían una vez y produce alcances menores; las conclusiones cualitativas (centralidad > azar, líderes de facción como semillas) se mantienen.

In [ ]:
def cascada_independiente(G, semillas, p=0.3, rondas=5, rng=None):
    informados, frontera = set(semillas), set(semillas)
    for _ in range(rondas):
        nuevos = set()
        for u in frontera:
            for v in G.neighbors(u):
                if v not in informados and v not in nuevos and rng.random() < p:
                    nuevos.add(v)
        informados |= nuevos; frontera = nuevos
    return len(informados)

rng = np.random.default_rng(3)
ic_tabla = {nombre: np.mean([cascada_independiente(G, s, rng=rng) for _ in range(N_SIM)]) for nombre, s in estrategias.items()}
ic_tabla["azar"] = np.mean([cascada_independiente(G, list(rng.choice(len(G), 3, replace=False)), rng=rng) for _ in range(N_SIM)])
pd.Series(ic_tabla, name="alcance medio (cascada independiente)").round(2)

## Ejercicio 2 · Eliminar el nodo con mayor intermediación

In [ ]:
nodo_critico = centralidades["intermediación"].idxmax()
H = G.copy(); H.remove_node(nodo_critico)
componentes = sorted(nx.connected_components(H), key=len, reverse=True)
gigante = H.subgraph(componentes[0])

print(f"Nodo eliminado: {nodo_critico} (intermediación = {centralidades.loc[nodo_critico, 'intermediación']:.3f})")
print(f"Componentes conexas: antes 1 → después {len(componentes)} de tamaños {[len(c) for c in componentes]}")
print("Nodos aislados del resto:", [sorted(c) for c in componentes[1:]])
print(f"Distancia media: antes {nx.average_shortest_path_length(G):.2f} → después (componente gigante) "
      f"{nx.average_shortest_path_length(gigante):.2f}")
print(f"Pares de nodos que siguen pudiendo comunicarse: "
      f"{sum(len(c) * (len(c) - 1) / 2 for c in componentes) / (len(H) * (len(H) - 1) / 2):.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dibujar(G, ["red" if n == nodo_critico else "lightblue" for n in G], titulo="Red original (en rojo, el nodo eliminado)", ax=axes[0], cmap=None)
color_comp = {n: i for i, c in enumerate(componentes) for n in c}
dibujar(H, [color_comp[n] for n in H], titulo=f"Sin el nodo {nodo_critico}: {len(componentes)} componentes", ax=axes[1], cmap="tab10")
plt.show()

**Ojo con la distancia media:** `nx.average_shortest_path_length` da error si el grafo no es conexo (hay pares a distancia infinita). Por eso la calculamos en la componente gigante. ¡Y baja! No porque la red haya mejorado, sino porque los nodos periféricos que colgaban del instructor ya no cuentan. Una métrica más honesta es la **eficiencia global** (media de $1/d_{ij}$, que vale 0 para pares desconectados):

In [ ]:
print(f"Eficiencia global: antes {nx.global_efficiency(G):.3f} → después {nx.global_efficiency(H):.3f}")

### Ampliación: ataque dirigido vs. fallos aleatorios
Eliminamos nodos uno a uno (hasta 10) y medimos el tamaño de la componente gigante. En el **ataque dirigido** quitamos cada vez el nodo de mayor intermediación (recalculada tras cada baja); en los **fallos aleatorios**, nodos al azar (media de 200 repeticiones).

In [ ]:
def tam_gigante(H):
    return max(len(c) for c in nx.connected_components(H)) if len(H) else 0

def ataque_dirigido(G, k):
    H = G.copy(); curva = [tam_gigante(H)]
    for _ in range(k):
        b = nx.betweenness_centrality(H)
        H.remove_node(max(b, key=b.get)); curva.append(tam_gigante(H))
    return curva

def fallos_aleatorios(G, k, reps=200, seed=0):
    rng = np.random.default_rng(seed); curvas = []
    for _ in range(reps):
        H = G.copy(); orden = rng.permutation(list(G)); curva = [tam_gigante(H)]
        for n in orden[:k]:
            H.remove_node(n); curva.append(tam_gigante(H))
        curvas.append(curva)
    return np.mean(curvas, axis=0)

K = 10
curva_ataque = ataque_dirigido(G, K)
curva_azar = fallos_aleatorios(G, K)
plt.plot(range(K + 1), curva_ataque, "o-", label="ataque dirigido (mayor intermediación)")
plt.plot(range(K + 1), curva_azar, "s--", label="fallos aleatorios (media)")
plt.xlabel("nodos eliminados"); plt.ylabel("tamaño de la componente gigante"); plt.legend(); plt.title("Robustez del club")
plt.show()
print("Ataque dirigido:", curva_ataque)
print("Fallos aleatorios:", np.round(curva_azar, 1).tolist())

**Interpretación.**

- Quitar **un solo nodo** (el 0, el instructor) parte la red en **3 componentes** (27, 5 y 1 nodos): los nodos 4, 5, 6, 10 y 16 y el nodo 11 solo se conectaban al club a través del instructor. Solo el **68 %** de los pares de personas puede seguir comunicándose y la eficiencia global cae de 0,49 a 0,36.
- La distancia media de la componente gigante *baja* (2,41 → 2,28): es un artefacto de medir solo a los que quedan conectados, no una mejora.
- La red es **robusta a fallos aleatorios** pero **frágil ante ataques dirigidos**: tras quitar 4 nodos elegidos por intermediación, la componente gigante tiene 10 nodos; quitando 4 al azar conserva ≈ 29. Es el comportamiento típico de redes con *hubs*.

**Implicación para una cadena de suministro.** Si los nodos son proveedores, almacenes y plantas, y las aristas rutas de suministro:
1. Los nodos de **alta intermediación** son puntos únicos de fallo (un puerto, un centro logístico, un proveedor exclusivo de un componente). Su caída no solo retrasa: puede **desconectar** clientes o plantas.
2. Medidas: identificar esos nodos, crear **rutas redundantes** (una segunda arista para los nodos que dependen de uno solo), contratar **proveedores alternativos** y dimensionar stock de seguridad aguas abajo del nodo crítico.
3. La métrica de seguimiento no debe ser la distancia media (que puede «mejorar» engañosamente), sino el **% de demanda que sigue conectada** o la eficiencia global.

## Ejercicio 3 · Adamic-Adar vs. vecinos comunes vs. Jaccard (20 semillas)

Repetimos el experimento del notebook: ocultamos el 10 % de las aristas, puntuamos todos los pares no conectados y contamos cuántas aristas ocultas aparecen en el top-$k$ ($k$ = nº de aristas ocultas). Además del número de aciertos (precisión@k), calculamos el **AUC**: probabilidad de que una arista oculta reciba más puntuación que un par que realmente no está conectado (0,5 = azar), que es menos ruidosa que contar aciertos en un top-7.

In [ ]:
from sklearn.metrics import roc_auc_score

def vecinos_comunes(G, pares):
    return [(u, v, len(list(nx.common_neighbors(G, u, v)))) for u, v in pares]

metodos = {
    "Adamic-Adar": nx.adamic_adar_index,
    "Vecinos comunes": vecinos_comunes,
    "Jaccard": nx.jaccard_coefficient,
}

def experimento(seed):
    rng = np.random.default_rng(seed)
    aristas = list(G.edges())
    ocultas = [aristas[i] for i in rng.choice(len(aristas), len(aristas) // 10, replace=False)]
    G_train = G.copy(); G_train.remove_edges_from(ocultas)
    ocultas_set = {(min(u, v), max(u, v)) for u, v in ocultas}
    no_aristas = list(nx.non_edges(G_train))
    # Desempates aleatorios: barajamos los pares antes de ordenar (importante para "vecinos comunes", con muchos empates)
    orden = rng.permutation(len(no_aristas))
    no_aristas = [no_aristas[i] for i in orden]
    fila = {"semilla": seed}
    for nombre, f in metodos.items():
        punt = list(f(G_train, no_aristas))
        ranking = sorted(punt, key=lambda t: -t[2])
        top = {(min(u, v), max(u, v)) for u, v, _ in ranking[:len(ocultas)]}
        y = [(min(u, v), max(u, v)) in ocultas_set for u, v, _ in punt]
        fila[f"{nombre} · aciertos"] = len(top & ocultas_set)
        fila[f"{nombre} · AUC"] = roc_auc_score(y, [s for _, _, s in punt])
    fila["azar · aciertos esperados"] = len(ocultas) ** 2 / len(no_aristas)
    return fila

resultados = pd.DataFrame([experimento(s) for s in range(20)]).set_index("semilla")
resumen = resultados.agg(["mean", "std"]).T.round(3)
resumen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
resultados[[c for c in resultados if "aciertos" in c and "azar" not in c]].boxplot(ax=axes[0])
axes[0].axhline(resultados["azar · aciertos esperados"].mean(), color="red", ls="--", label="azar"); axes[0].legend()
axes[0].set_title("Aristas ocultas recuperadas en el top-7 (20 semillas)")
resultados[[c for c in resultados if "AUC" in c]].boxplot(ax=axes[1])
axes[1].axhline(0.5, color="red", ls="--"); axes[1].set_title("AUC (20 semillas)")
for ax in axes: ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

In [ ]:
mejor_auc = resumen.loc[[i for i in resumen.index if "AUC" in i], "mean"].idxmax()
print("Método con mayor AUC medio:", mejor_auc)

**Interpretación.**

- **Adamic-Adar** es el mejor método de media: recupera ≈ 0,75 de las 7 aristas ocultas en el top-7 y tiene AUC ≈ 0,72. **Vecinos comunes** queda muy cerca (≈ 0,70 aciertos, AUC ≈ 0,68): ambas premian tener muchos amigos en común, y Adamic-Adar además descuenta a los amigos comunes muy populares. El azar recuperaría ≈ 0,1 aristas (AUC 0,5).
- **Jaccard** no recupera **ninguna** arista oculta en el top-7 en ninguna de las 20 semillas (AUC ≈ 0,60). Al dividir por la *unión* de vecindarios, sus puntuaciones más altas van a pares de nodos periféricos con 1–2 contactos que comparten casi todo su (pequeño) vecindario, mientras que muchas aristas ocultas tocan a los *hubs* (0, 33), cuya unión de vecindarios es enorme.
- La **desviación típica** de los aciertos (≈ 0,65) es tan grande como la media: con solo 7 aristas ocultas, una sola semilla no basta para decidir entre Adamic-Adar y vecinos comunes. Por eso hay que repetir con varias semillas y conviene mirar una métrica menos ruidosa como el AUC.
- En negocio (recomendar contactos, detectar relaciones ocultas entre empresas o cuentas), lo razonable es probar varias heurísticas con validación repetida y usarlas después como *features* de un clasificador supervisado.

## Ejercicio 4 · ¿Quién es el personaje más central en *Los miserables*?

In [ ]:
LM = nx.les_miserables_graph()
print(f"Personajes: {LM.number_of_nodes()} | Relaciones (coapariciones): {LM.number_of_edges()}")

cent_lm = pd.DataFrame({
    "grado": nx.degree_centrality(LM),
    "intermediación": nx.betweenness_centrality(LM),
    "cercanía": nx.closeness_centrality(LM),
    "autovector": nx.eigenvector_centrality(LM, max_iter=1000),
    "pagerank": nx.pagerank(LM),   # usa el peso (nº de coapariciones), como en el notebook
})
top_lm = pd.DataFrame({m: cent_lm[m].sort_values(ascending=False).index[:5] for m in cent_lm.columns},
                      index=[f"#{i}" for i in range(1, 6)])
top_lm

In [ ]:
print("Correlación de Spearman entre rankings:")
cent_lm.corr(method="spearman").round(2)

In [ ]:
pos_lm = nx.spring_layout(LM, seed=1)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, m in zip(axes, ["intermediación", "autovector"]):
    v = cent_lm[m]
    nx.draw_networkx(LM, pos_lm, node_size=3000 * v / v.max() + 20, node_color=list(v), cmap="viridis",
                     font_size=7, edge_color="lightgray", ax=ax,
                     labels={n: n for n in v.sort_values(ascending=False).index[:8]})
    ax.set_title(f"Los miserables · tamaño = {m}"); ax.axis("off")
plt.show()

**Interpretación.**

- **Valjean** es el más central según grado, intermediación, cercanía y PageRank: es el protagonista y aparece con casi todos los grupos de la novela.
- Con **autovector** el primero es **Gavroche** (Valjean queda segundo): Gavroche está metido en el grupo más denso y cohesionado de la novela (los estudiantes de la barricada: Enjolras, Courfeyrac, Combeferre...), y la centralidad de autovector premia estar rodeado de nodos que a su vez están muy conectados entre sí.
- En **intermediación** aparece **Myriel** (el obispo) en el top-3 aunque tiene poca relevancia en la trama global: es el único enlace de un grupo de personajes secundarios (Napoleón, la condesa de Lô, Cravatte...) con el resto de la red. Es el ejemplo de manual de **intermediario** o «puente».
- Conclusión: las medidas **no coinciden del todo** porque responden a preguntas distintas (¿quién tiene más relaciones?, ¿quién conecta grupos?, ¿quién está cerca de todos?, ¿quién está en el núcleo más influyente?). Elegir la centralidad es elegir la pregunta de negocio.